In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

import data
from check_submission import check_all

ROOT = Path.cwd()
TRACKS = ["off_market", "list_aware"]
# Walk-forward validation: each of these months is predicted by models trained only
# on the sales before that month, the same way the real test works.
VALIDATION_MONTHS = pd.period_range("2025-12", "2026-05", freq="M")


In [2]:
def validate(make_model_func):

    listings = data.load("listings")
    results = []
    model_list = []
    for month in VALIDATION_MONTHS:
        train = listings[listings["date_sold"] < month.start_time]
        homes = listings[listings["date_sold"].dt.to_period("M") == month]
        model_sublist = []
        for track in TRACKS:
            model = make_model_func(track).fit(train)
            model_sublist.append(model)
            predicted = model.predict(data.hide_answers(homes, track))
            results.append(pd.DataFrame({
                "ml_num": homes["ml_num"], "track": track, "month": str(month),
                "house_type_name": homes["house_type_name"], "sold_price": homes["sold_price"],
                "predicted_price": np.round(predicted)}))
        print(f"{month}: trained on {len(train):,} earlier sales, predicted {len(homes):,} homes")
        model_list.append(model_sublist)
    return pd.concat(results, ignore_index=True), model_list

def bootstrap_ci(values, statistic, repeats=1000):
    """95% interval: recompute the statistic on 1,000 resamples of the homes
    (drawn with replacement) and keep the middle 95% of the results."""
    rng = np.random.default_rng(0)
    results = [statistic(rng.choice(values, size=len(values))) for _ in range(repeats)]
    return [round(float(x), 2) for x in np.percentile(results, [2.5, 97.5])]


def metrics(predictions):
    """Scores for one track. Errors are in percent of the sold price."""
    signed = 100 * (predictions["predicted_price"] - predictions["sold_price"]) / predictions["sold_price"]
    absolute = signed.abs().to_numpy()
    return {
        "n": len(predictions),
        "mape": round(float(np.mean(absolute)), 2),          # mean absolute % error
        "mape_95ci": bootstrap_ci(absolute, np.mean),
        "mdape": round(float(np.median(absolute)), 2),       # median absolute % error
        "mdape_95ci": bootstrap_ci(absolute, np.median),
        "median_signed_error": round(float(np.median(signed)), 2),  # above 0: predicts too high
    }
def report(df):
    # Accept results from earlier notebook runs without refitting the models.
    predictions = pd.concat(df, ignore_index=True) if isinstance(df, list) else df
    scores = {track: metrics(predictions[predictions["track"] == track]) for track in TRACKS}

    (ROOT / "reports").mkdir(exist_ok=True)
    (ROOT / "reports" / "validation_report.json").write_text(json.dumps(scores, indent=2))
    lines = [
        "# Validation report", "",
        f"Walk-forward validation, {VALIDATION_MONTHS[0]} to {VALIDATION_MONTHS[-1]}: each month was "
        "predicted by models trained only on the sales before it.",
        "Errors are (predicted - sold) / sold. Brackets: bootstrap 95% interval.", "",
        "| Track | Homes | MAPE | MdAPE | Median signed error |", "|---|---|---|---|---|"]
    for track, s in scores.items():
        lines.append(f"| {track} | {s['n']:,} | {s['mape']:.2f}% ({s['mape_95ci'][0]:.2f} to {s['mape_95ci'][1]:.2f}) "
                     f"| {s['mdape']:.2f}% ({s['mdape_95ci'][0]:.2f} to {s['mdape_95ci'][1]:.2f}) "
                     f"| {s['median_signed_error']:+.2f}% |")
    lines += [
        "", "MAPE: mean absolute percentage error. MdAPE: median absolute percentage error.",
        "A median signed error above 0 means the model tends to predict too high.",
        "The intervals treat homes as independent. Homes sold in the same month share the same",
        "market swings, so the real uncertainty is somewhat wider.", "",
        "![Error by price range](../figures/error_by_price_range.png)", ""]
    (ROOT / "reports" / "validation_report.md").write_text("\n".join(lines))
    write_latex_table(scores)
    error_by_price_range(predictions)
    print("\n".join(lines))
    print("Wrote reports/validation_report.md, reports/validation_report.json, reports/results_table.tex "
          "and figures/error_by_price_range.png")


def write_latex_table(scores):
    """The same scores as a LaTeX table, for report/report.tex to include."""
    rows = []
    for track, s in scores.items():
        name = track.replace("_", "-")
        rows.append(f"{name} & {s['n']:,} & {s['mdape']:.2f} ({s['mdape_95ci'][0]:.2f}--{s['mdape_95ci'][1]:.2f}) "
                    f"& {s['mape']:.2f} ({s['mape_95ci'][0]:.2f}--{s['mape_95ci'][1]:.2f}) "
                    f"& {s['median_signed_error']:+.2f} \\\\")
    table = "\n".join([
        "% Written by `python run.py report`. Do not edit by hand: rerun the command instead.",
        "\\begin{tabular}{lrccc}", "\\hline",
        "Track & Homes & MdAPE \\% (95\\% CI) & MAPE \\% (95\\% CI) & Median signed error \\% \\\\", "\\hline",
        *rows, "\\hline", "\\end{tabular}"])
    (ROOT / "reports" / "results_table.tex").write_text(table + "\n")


def error_by_price_range(predictions):
    """Figure: how big the errors are, from the cheapest tenth of homes to the dearest.

    Want more figures? Copy this function and group the errors by something else:
    house_type_name, month, or community (add it to the predictions file in validate()).
    """
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(8, 4.5))
    colours = {"off_market": "#2a78d6", "list_aware": "#eb6834"}
    for track in TRACKS:
        rows = predictions[predictions["track"] == track]
        error = 100 * (rows["predicted_price"] - rows["sold_price"]).abs() / rows["sold_price"]
        decile = pd.qcut(rows["sold_price"], 10, labels=False) + 1   # 1 = cheapest tenth
        by_decile = error.groupby(decile)
        middle = by_decile.median()
        ax.plot(middle.index, middle.values, marker="o", linewidth=2, color=colours[track], label=track)
        ax.fill_between(middle.index, by_decile.quantile(0.25), by_decile.quantile(0.75),
                        color=colours[track], alpha=0.15, linewidth=0)
    prices = rows["sold_price"].groupby(decile).median()
    ax.set_xticks(prices.index, [f"{d}\n${p / 1e6:.2f}M" for d, p in prices.items()])
    ax.set_xlabel("Sold-price decile (1 = cheapest tenth of homes), with its median price")
    ax.set_ylabel("Absolute % error\n(line: median, band: middle 50%)")
    ax.set_title(f"Validation error by price range, {VALIDATION_MONTHS[0]} to {VALIDATION_MONTHS[-1]}")
    ax.set_ylim(bottom=0)
    ax.grid(axis="y", color="#e5e5e5")
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False)
    fig.tight_layout()
    (ROOT / "figures").mkdir(exist_ok=True)
    fig.savefig(ROOT / "figures" / "error_by_price_range.png", dpi=150)
    plt.close(fig)

In [3]:
import models.gb_models as gb_models

predictions, models = validate(gb_models.make_model)

ARIMA converged: True
Iterations: 3
Parameters: ar.L1    -0.485570
sigma2    0.000501
dtype: float64


c:\Users\deant\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


ARIMA converged: False
Iterations: 12
Parameters: ar.L1    -0.539715
sigma2    0.000808
dtype: float64
ARIMA converged: True
Iterations: 15
Parameters: ar.L1    -0.558799
sigma2    0.000690
dtype: float64
ARIMA converged: True
Iterations: 12
Parameters: ar.L1    -0.522866
sigma2    0.000614
dtype: float64
ARIMA converged: True
Iterations: 3
Parameters: ar.L1    -0.485570
sigma2    0.000501
dtype: float64
2025-12: trained on 139,931 earlier sales, predicted 1,238 homes
ARIMA converged: True
Iterations: 17
Parameters: ar.L1    -0.529283
sigma2    0.000567
dtype: float64
ARIMA converged: True
Iterations: 18
Parameters: ar.L1    -0.576826
sigma2    0.000747
dtype: float64
ARIMA converged: True
Iterations: 3
Parameters: ar.L1    -0.543137
sigma2    0.000580
dtype: float64
ARIMA converged: True
Iterations: 18
Parameters: ar.L1    -0.662901
sigma2    0.000712
dtype: float64
ARIMA converged: True
Iterations: 17
Parameters: ar.L1    -0.529283
sigma2    0.000567
dtype: float64
2026-01: trained o

c:\Users\deant\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


ARIMA converged: False
Iterations: 16
Parameters: ar.L1    -0.607086
sigma2    0.000794
dtype: float64
ARIMA converged: True
Iterations: 15
Parameters: ar.L1    -0.524496
sigma2    0.000796
dtype: float64
ARIMA converged: True
Iterations: 15
Parameters: ar.L1    -0.540401
sigma2    0.000386
dtype: float64
2026-02: trained on 142,227 earlier sales, predicted 1,442 homes
ARIMA converged: True
Iterations: 16
Parameters: ar.L1    -0.553237
sigma2    0.000500
dtype: float64
ARIMA converged: True
Iterations: 16
Parameters: ar.L1    -0.574713
sigma2    0.000892
dtype: float64
ARIMA converged: True
Iterations: 20
Parameters: ar.L1    -0.714368
sigma2    0.000589
dtype: float64


c:\Users\deant\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


ARIMA converged: False
Iterations: 17
Parameters: ar.L1    -0.653424
sigma2    0.000661
dtype: float64
ARIMA converged: True
Iterations: 16
Parameters: ar.L1    -0.553237
sigma2    0.000500
dtype: float64
2026-03: trained on 143,669 earlier sales, predicted 1,816 homes
ARIMA converged: True
Iterations: 15
Parameters: ar.L1    -0.615626
sigma2    0.000429
dtype: float64
ARIMA converged: True
Iterations: 15
Parameters: ar.L1    -0.621707
sigma2    0.000892
dtype: float64
ARIMA converged: True
Iterations: 17
Parameters: ar.L1    -0.609013
sigma2    0.000565
dtype: float64
ARIMA converged: True
Iterations: 16
Parameters: ar.L1    -0.496626
sigma2    0.000477
dtype: float64
ARIMA converged: True
Iterations: 15
Parameters: ar.L1    -0.615626
sigma2    0.000429
dtype: float64
2026-04: trained on 145,485 earlier sales, predicted 2,224 homes
ARIMA converged: True
Iterations: 14
Parameters: ar.L1    -0.574094
sigma2    0.000439
dtype: float64
ARIMA converged: True
Iterations: 12
Parameters: ar.L

In [4]:
report(predictions)

# Validation report

Walk-forward validation, 2025-12 to 2026-05: each month was predicted by models trained only on the sales before it.
Errors are (predicted - sold) / sold. Brackets: bootstrap 95% interval.

| Track | Homes | MAPE | MdAPE | Median signed error |
|---|---|---|---|---|
| off_market | 10,069 | 12.67% (12.46 to 12.88) | 10.02% (9.78 to 10.27) | +0.38% |
| list_aware | 10,069 | 5.82% (5.73 to 5.91) | 4.73% (4.62 to 4.84) | -2.34% |

MAPE: mean absolute percentage error. MdAPE: median absolute percentage error.
A median signed error above 0 means the model tends to predict too high.
The intervals treat homes as independent. Homes sold in the same month share the same
market swings, so the real uncertainty is somewhat wider.

![Error by price range](../figures/error_by_price_range.png)

Wrote reports/validation_report.md, reports/validation_report.json, reports/results_table.tex and figures/error_by_price_range.png
